### Feature Engineering

In [1]:
import json
import os
import pandas as pd
from pyopensky.rest import REST

# 1. Connexion à l'API et extraction
with open("credentials.json", "r") as f:
    creds = json.load(f)

os.environ["OPENSKY_CLIENT_ID"] = creds["clientId"]
os.environ["OPENSKY_CLIENT_SECRET"] = creds["clientSecret"]

print("Extraction des données OpenSky en cours...")
rest = REST()
df = rest.states()

# 2. Création d'une colonne altitude UNIQUE (priorité au GPS, fallback sur le barométrique)
if 'geo_altitude' in df.columns and 'baro_altitude' in df.columns:
    df['altitude'] = df['geo_altitude'].fillna(df['baro_altitude'])
elif 'geo_altitude' in df.columns:
    df['altitude'] = df['geo_altitude']

# 3. Renommage strict sans créer de doublons
df = df.rename(columns={
    'velocity': 'groundspeed',
    'true_track': 'track',
    'truetrack': 'track',
    'verticalrate': 'vertical_rate'
})

# 4. Sécurité finale : on supprime les éventuelles colonnes en double
df = df.loc[:, ~df.columns.duplicated()]

print(f"Extraction réussie : {len(df)} vols chargés dans la variable 'df'.")

Extraction des données OpenSky en cours...
Extraction réussie : 7905 vols chargés dans la variable 'df'.


In [2]:
import numpy as np
import pandas as pd
from scipy.spatial import KDTree

# 1. Filtre géographique régional (ex: Europe) pour éviter la distorsion cartésienne
df_atc = df[
    (df['onground'] == False) & 
    (df['latitude'].between(35.0, 70.0)) & 
    (df['longitude'].between(-15.0, 30.0))
].copy()

# Nettoyage et typage
cols_num = ['longitude', 'latitude', 'altitude', 'groundspeed', 'track', 'vertical_rate']
for col in cols_num:
    if col in df_atc.columns:
        df_atc[col] = df_atc[col].astype('float64')

df_atc = df_atc.dropna(subset=['latitude', 'longitude', 'altitude', 'groundspeed', 'track']).reset_index(drop=True)

# 2. Vectorisation 3D
track_rad = np.radians(df_atc['track'])
df_atc['vx'] = df_atc['groundspeed'] * np.sin(track_rad)
df_atc['vy'] = df_atc['groundspeed'] * np.cos(track_rad)
df_atc['vz'] = df_atc['vertical_rate'].fillna(0)

# 3. Projection Cartésienne Locale (stabilisée par le filtre régional)
R = 6371000.0
lat_mean = np.radians(df_atc['latitude'].mean())
df_atc['x_2d'] = R * np.radians(df_atc['longitude']) * np.cos(lat_mean)
df_atc['y_2d'] = R * np.radians(df_atc['latitude'])

# 4. KDTree pour cibler les voisinages (rayon élargi pour laisser le temps au RL d'agir)
search_radius_m = 20.0 * 1852.0  # 20 NM
coords_2d = np.column_stack((df_atc['x_2d'], df_atc['y_2d']))
tree = KDTree(coords_2d)
pairs = tree.query_pairs(r=search_radius_m)

# 5. Extraction des vecteurs d'états pour le RL (CPA Horizontal)
rl_state_vectors = []
cpa_horizon_sec = 180.0

for i, j in pairs:
    r1, r2 = df_atc.iloc[i], df_atc.iloc[j]
    
    # Vecteur de position relative
    dx = r1['x_2d'] - r2['x_2d']
    dy = r1['y_2d'] - r2['y_2d']
    dz = r1['altitude'] - r2['altitude']
    
    # Vecteur de vitesse relative
    dvx = r1['vx'] - r2['vx']
    dvy = r1['vy'] - r2['vy']
    dvz = r1['vz'] - r2['vz']
    
    # Calcul du CPA purement horizontal (X, Y)
    dv2_horiz = dvx**2 + dvy**2
    if dv2_horiz > 1e-6:
        # Instant du croisement le plus proche au sol
        t_cpa = -(dx * dvx + dy * dvy) / dv2_horiz
        
        if 0 < t_cpa <= cpa_horizon_sec:
            # Projection des positions à l'instant t_cpa
            rx_cpa = dx + dvx * t_cpa
            ry_cpa = dy + dvy * t_cpa
            rz_cpa = dz + dvz * t_cpa  # Différence d'altitude au moment critique
            
            d_cpa_horiz_nm = np.hypot(rx_cpa, ry_cpa) / 1852.0
            d_cpa_vert_m = abs(rz_cpa)
            
            # Filtre : On ne garde que les paires qui brisent les normes de séparation ATC
            if d_cpa_horiz_nm < 5.0 and d_cpa_vert_m < 304.8:
                rl_state_vectors.append({
                    'agent_callsign': r1['callsign'],
                    'intruder_callsign': r2['callsign'],
                    'dx': dx, 'dy': dy, 'dz': dz,
                    'dvx': dvx, 'dvy': dvy, 'dvz': dvz,
                    't_cpa': t_cpa,
                    'd_cpa_horiz_nm': d_cpa_horiz_nm,
                    'd_cpa_vert_m': d_cpa_vert_m
                })

# Création du dataset final prêt pour l'environnement PyTorch
df_rl_states = pd.DataFrame(rl_state_vectors)
print(f"Scénarios de conflit générés pour l'entraînement RL : {len(df_rl_states)}")

# Affichage des autres métriques
if not df_rl_states.empty:
    print("\n=== VECTEURS D'ÉTAT (Aperçu) ===")
    print(df_rl_states[['agent_callsign', 'intruder_callsign', 'dx', 'dy', 'dz', 'dvx', 'dvy', 'dvz']].head())
    
    print("\n=== MÉTRIQUES DE SÉVÉRITÉ (CPA) ===")
    print(df_rl_states[['t_cpa', 'd_cpa_horiz_nm', 'd_cpa_vert_m']].describe().round(2))
else:
    print("Aucun conflit détecté sur cet échantillon.")

Scénarios de conflit générés pour l'entraînement RL : 46

=== VECTEURS D'ÉTAT (Aperçu) ===
  agent_callsign intruder_callsign            dx            dy       dz  \
0          GMACI           RYR77DU -12915.852823  14255.189596  -678.18   
1         VJT464           IAM4188    -82.218971   9407.090794  -335.28   
2          GEFSD             GBTJA   4043.678459  -8284.022035  -243.84   
3         LOT5YH            LOT2LQ  25876.552357 -10218.813759 -2613.66   
4        EZY3298           RYR22JD     -7.474452      0.000000     7.62   

          dvx         dvy    dvz  
0   85.402302 -157.927233   7.80  
1   36.518928  -80.767676   1.95  
2   32.922640   22.116336  -0.98  
3 -252.589816   84.890781  24.71  
4    6.682526    3.600197  -0.32  

=== MÉTRIQUES DE SÉVÉRITÉ (CPA) ===
        t_cpa  d_cpa_horiz_nm  d_cpa_vert_m
count   46.00           46.00         46.00
mean    62.30            2.23        141.53
std     45.51            1.74        112.82
min      0.87            0.00      

In [3]:
%pip install scikit-learn
%pip install torch


[notice] A new release of pip is available: 25.3 -> 26.3
[notice] To update, run: /opt/homebrew/opt/python@3.11/bin/python3.11 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 25.3 -> 26.3
[notice] To update, run: /opt/homebrew/opt/python@3.11/bin/python3.11 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [4]:
from sklearn.ensemble import IsolationForest

# 1. Règles métier ATC : Appliquées sur nos paires (Agent vs Intruder)
# Calcul de la distance instantanée (à t=0) à partir des vecteurs x, y, z
df_rl_states['dist_instant_nm'] = np.hypot(df_rl_states['dx'], df_rl_states['dy']) / 1852.0
df_rl_states['delta_alt_instant_m'] = df_rl_states['dz'].abs()

# Conflit instantané : les normes sont DÉJÀ violées à l'instant T
df_rl_states['is_conflict_instant'] = (
    (df_rl_states['dist_instant_nm'] < 5.0) & 
    (df_rl_states['delta_alt_instant_m'] < 304.8)
)

# Conflit prédictif : les normes seront violées au CPA (dans moins de 3 min)
df_rl_states['is_conflict_predictive'] = (
    (df_rl_states['d_cpa_horiz_nm'] < 5.0) & 
    (df_rl_states['d_cpa_vert_m'] < 304.8) &
    (df_rl_states['t_cpa'] <= 180.0) &
    (~df_rl_states['is_conflict_instant']) # On exclut ceux déjà en conflit
)

# 2. Modèle ML d'anomalies sur la cinématique du conflit
# On cherche les situations géométriques bizarres (artefacts API)
features_ml = ['dx', 'dy', 'dz', 'dvx', 'dvy', 'dvz', 't_cpa']

if len(df_rl_states) > 0:
    X = df_rl_states[features_ml].fillna(df_rl_states[features_ml].median())
    
    # Contamination ajustée (ex: 5%) car le dataset de paires est plus petit
    iso = IsolationForest(contamination=0.05, random_state=42)
    df_rl_states['anomaly_score'] = iso.fit_predict(X)
    df_rl_states['is_anomaly'] = df_rl_states['anomaly_score'] == -1
else:
    df_rl_states['is_anomaly'] = False

# 3. Synthèse globale pour le système RL
print("=== SYNTHÈSE DU SYSTÈME ATC (SCÉNARIOS RL) ===")
print(f"Voisinages initiaux détectés (KDTree) : {len(pairs)}")
print(f"Conflits critiques retenus            : {len(df_rl_states)}")
print(f" ┣━ Conflits instantanés (à corriger d'urgence) : {df_rl_states['is_conflict_instant'].sum()}")
print(f" ┗━ Conflits prédictifs (anticipation < 3 min)  : {df_rl_states['is_conflict_predictive'].sum()}")
print(f"Géométries aberrantes isolées par le ML (5%)    : {df_rl_states['is_anomaly'].sum()}")

# Extraire et afficher les vols pour l'environnement PyTorch
alertes = df_rl_states[df_rl_states['is_conflict_predictive'] | df_rl_states['is_anomaly']]
print("\n=== EXTRAIT DES SCÉNARIOS À HAUTE PRIORITÉ ===")
if not alertes.empty:
    print(alertes[['agent_callsign', 'intruder_callsign', 't_cpa', 'dist_instant_nm', 'is_conflict_predictive', 'is_anomaly']].head(10))
else:
    print("Aucun scénario prioritaire à afficher.")

=== SYNTHÈSE DU SYSTÈME ATC (SCÉNARIOS RL) ===
Voisinages initiaux détectés (KDTree) : 4559
Conflits critiques retenus            : 46
 ┣━ Conflits instantanés (à corriger d'urgence) : 29
 ┗━ Conflits prédictifs (anticipation < 3 min)  : 17
Géométries aberrantes isolées par le ML (5%)    : 3

=== EXTRAIT DES SCÉNARIOS À HAUTE PRIORITÉ ===
   agent_callsign intruder_callsign       t_cpa  dist_instant_nm  \
0           GMACI           RYR77DU  104.059922        10.386693   
1          VJT464           IAM4188   97.083696         5.079617   
3          LOT5YH            LOT2LQ  104.264737        15.022255   
5          HZSKY1            DLH766   68.656684         5.076993   
6          LOT2VL            SN81XP  173.450595         9.930640   
7          WMT2TS             WZZ87   40.428241         8.241285   
9                             HBXDA   19.958228         5.073428   
16         SN81XP           SWR139K   84.452101         5.923822   
17        DEMON01           GNSHP02  151.412830

## Converting In Tensor

In [5]:
import torch
from sklearn.preprocessing import StandardScaler

# 1. Exclusion des anomalies radar repérées par le Machine Learning
df_clean = df_rl_states[~df_rl_states['is_anomaly']].copy().reset_index(drop=True)

# 2. Sélection stricte du vecteur d'état (les yeux de notre agent IA)
state_features = ['dx', 'dy', 'dz', 'dvx', 'dvy', 'dvz']

# 3. Normalisation statistique (Z-score)
scaler = StandardScaler()
states_scaled = scaler.fit_transform(df_clean[state_features])

# 4. Conversion en tenseur PyTorch (format float32 requis pour les poids du DQN)
initial_states_tensor = torch.tensor(states_scaled, dtype=torch.float32)

print("=== PRÉPARATION DES TENSEURS PYTORCH ===")
print(f"Scénarios valides retenus pour l'entraînement : {len(df_clean)}")
print(f"Format de la matrice d'états : {initial_states_tensor.shape}")
print(f"\nExemple du premier état normalisé que le réseau lira au reset() :\n{initial_states_tensor[0]}")

=== PRÉPARATION DES TENSEURS PYTORCH ===
Scénarios valides retenus pour l'entraînement : 43
Format de la matrice d'états : torch.Size([43, 6])

Exemple du premier état normalisé que le réseau lira au reset() :
tensor([ 0.0529,  1.4562, -1.5383,  0.0427, -0.6908,  1.0687])


## Enregistrement du tenseur : initial_states_tensor, "initial_states.pt"

In [6]:
torch.save(initial_states_tensor, "initial_states.pt")
print("Tenseur sauvegardé avec succès !")

Tenseur sauvegardé avec succès !
